In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from catboost import Pool
import optuna
from collections import defaultdict
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import TargetEncoder, LabelEncoder
from sklearn.utils.validation import check_is_fitted
# from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
from itertools import combinations
from sklearn.metrics import roc_auc_score
from typing import List, Union, Optional
import pickle
import joblib
import json
from pathlib import Path
import time
import copy
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/cdeotte/s6e4-original-dataset/Heart_Disease_Prediction.csv
/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [2]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

train = pd.read_csv(f'{CONFIG.INPUT_DIR}/train.csv')
train['source'] = 'train'
test = pd.read_csv(f'{CONFIG.INPUT_DIR}/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv(f'{CONFIG.INPUT_DIR}/sample_submission.csv')

org = pd.read_csv('/kaggle/input/datasets/cdeotte/s6e4-original-dataset/Heart_Disease_Prediction.csv')
org['source'] = 'original'

combine = pd.concat([train.drop(columns='id'), test.drop(columns='id'), org], ignore_index=True).reset_index()

In [3]:
NUMS = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
# NUMS = [col for col in NUMS if col not in BINS]
HIGH_CARDINALITY = [col for col in NUMS if train[col].nunique() > 40]

In [4]:
CATS = []
for c in NUMS:
    n = f'{c}_cat'
    combine[n] = combine[c].astype(str).astype('category')
    CATS.append(n)

print(CATS)
print('='*30)
print(len(CATS))

['Age_cat', 'Sex_cat', 'Chest pain type_cat', 'BP_cat', 'Cholesterol_cat', 'FBS over 120_cat', 'EKG results_cat', 'Max HR_cat', 'Exercise angina_cat', 'ST depression_cat', 'Slope of ST_cat', 'Number of vessels fluro_cat', 'Thallium_cat']
13


In [5]:
train = combine.loc[combine['source']=='train']
test = combine.loc[combine['source']=='test']
org = combine.loc[combine['source']=='original']

In [6]:
for df in [org, train, test]:
    int_cols = df.select_dtypes(include=['int64']).columns.to_list()
    float_cols = df.select_dtypes(include=['float64']).columns.to_list()
    df[int_cols] = df[int_cols].astype('int32')
    df[float_cols] = df[float_cols].astype('float32')

In [7]:
FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source', 'index', 'strat_feature']]
# FEATURES = [col for col in FEATURES if col not in INTER]
print(FEATURES)
print(len(FEATURES))

X = train[FEATURES]
# X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
# y_org = org[CONFIG.TARGET].map(class_mapping)

X_test = test[FEATURES]

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Age_cat', 'Sex_cat', 'Chest pain type_cat', 'BP_cat', 'Cholesterol_cat', 'FBS over 120_cat', 'EKG results_cat', 'Max HR_cat', 'Exercise angina_cat', 'ST depression_cat', 'Slope of ST_cat', 'Number of vessels fluro_cat', 'Thallium_cat']
26


In [8]:
skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)
kf = KFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)

strat_cols = ['Thallium', 'Chest pain type', 'Heart Disease']
le = LabelEncoder()
stratify_feature = le.fit_transform(train[strat_cols].astype(str).agg('_'.join, axis=1))

In [9]:
# ARTIFACT_DIR = Path('/kaggle/working/optuna_artifacts')
# ARTIFACT_DIR.mkdir(exist_ok=True)

# # Global storage for all trials
# trial_predictions = {}
# trial_oof_preds = {}
# trial_start_times = {}
# all_models_info = []
# completed_trials = 0

# # ===== SAVING FUNCTIONS =====
# def save_optuna_artifacts(force_save=False):
#     """Save all optuna artifacts to disk"""
#     global completed_trials
    
#     # Only save every 5 trials to avoid slowing down
#     if not force_save and completed_trials % 5 != 0:
#         return
    
#     try:
#         # Save trial predictions
#         with open(ARTIFACT_DIR / 'trial_predictions.pkl', 'wb') as f:
#             pickle.dump(trial_predictions, f)
        
#         # Save OOF predictions
#         with open(ARTIFACT_DIR / 'trial_oof_preds.pkl', 'wb') as f:
#             pickle.dump(trial_oof_preds, f)
        
#         # Save model info as CSV
#         if all_models_info:
#             models_df = pd.DataFrame(all_models_info)
#             models_df.to_csv(ARTIFACT_DIR / 'all_models_info.csv', index=False)
        
#         # Save predictions as numpy arrays for easy loading
#         if trial_predictions:
#             trial_numbers = sorted(trial_predictions.keys())
#             test_preds_list = [trial_predictions[t]['test_preds'] for t in trial_numbers]
#             oof_preds_list = [trial_oof_preds[t] for t in trial_numbers]
            
#             if test_preds_list:
#                 np.save(ARTIFACT_DIR / 'all_test_preds.npy', np.stack(test_preds_list))
#                 np.save(ARTIFACT_DIR / 'all_oof_preds.npy', np.stack(oof_preds_list))
        
#         # Save metadata
#         metadata = {
#             'n_trials': len(trial_predictions),
#             'completed_trials': completed_trials,
#             'save_time': time.strftime('%Y-%m-%d %H:%M:%S'),
#             'best_trial': max(trial_predictions.items(), key=lambda x: x[1]['oof_score'])[0] if trial_predictions else None,
#             'best_score': max([trial_predictions[t]['oof_score'] for t in trial_predictions]) if trial_predictions else 0
#         }
#         with open(ARTIFACT_DIR / 'metadata.json', 'w') as f:
#             json.dump(metadata, f, indent=2)
        
#         if force_save or completed_trials % 10 == 0:
#             print(f"💾 Saved artifacts for {len(trial_predictions)} trials")
            
#     except Exception as e:
#         print(f"⚠ Failed to save artifacts: {e}")

# # ===== OBJECTIVE FUNCTION WITH FIXES =====
# def objective(trial):
#     global completed_trials
    
#     # Track start time
#     trial_start = time.time()
#     trial_start_times[trial.number] = trial_start
    
#     print(f"\n{'='*70}")
#     print(f"TRIAL {trial.number} STARTING")
#     print(f"{'='*70}")
    
#     # ===== PARAMETER SAMPLING =====
#     booster = trial.suggest_categorical('booster', ['gbtree'])
#     print(f"Booster: {booster}")
    
#     learning_rate = trial.suggest_float('learning_rate', 0.0005, 0.05, log=True)
#     n_estimators = trial.suggest_int('n_estimators', 2000, 15000, step=500)
    
#     # Tree parameters (only for tree-based boosters)
#     if booster != 'gblinear':
#         max_depth = trial.suggest_int('max_depth', 4, 30)
#         gamma = trial.suggest_float('gamma', 0.0, 5.0)
#         min_child_weight = trial.suggest_float('min_child_weight', 1, 20)
#     else:
#         max_depth = 0
#         gamma = 0
#         min_child_weight = 1
    
#     # Regularization
#     reg_alpha = trial.suggest_float('reg_alpha', 1e-6, 8.0, log=True)
#     reg_lambda = trial.suggest_float('reg_lambda', 1e-6, 10.0, log=True)
    
#     # Sampling parameters
#     if booster in ['gbtree', 'dart']:
#         subsample = trial.suggest_float('subsample', 0.4, 1.0)
#         colsample_bytree = trial.suggest_float('colsample_bytree', 0.3, 1.0)
#         colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.3, 1.0)
#     else:
#         subsample = 1.0
#         colsample_bytree = 1.0
#         colsample_bylevel = 1.0
    
#     # DART-specific parameters
#     if booster == 'dart':
#         rate_drop = trial.suggest_float('rate_drop', 0.05, 0.3)
#         skip_drop = trial.suggest_float('skip_drop', 0.3, 0.7)
#         sample_type = trial.suggest_categorical('sample_type', ['uniform', 'weighted'])
#         normalize_type = trial.suggest_categorical('normalize_type', ['tree', 'forest'])
#     else:
#         rate_drop = 0.0
#         skip_drop = 0.0
#         sample_type = 'uniform'
#         normalize_type = 'tree'  # FIX: Added default for non-dart boosters
    
#     # ===== BUILD PARAMETERS =====
#     params = {
#         'booster': booster,
#         'learning_rate': learning_rate,
#         'n_estimators': n_estimators,
#         'max_depth': max_depth,
#         'min_child_weight': min_child_weight,
#         'reg_alpha': reg_alpha,
#         'reg_lambda': reg_lambda,
#         'gamma': gamma,
#         'subsample': subsample,
#         'colsample_bytree': colsample_bytree,
#         'colsample_bylevel': colsample_bylevel,
#         'rate_drop': rate_drop,
#         'skip_drop': skip_drop,
#         'sample_type': sample_type,
#         'normalize_type': normalize_type,
#         'objective': 'binary:logistic',
#         'eval_metric': 'auc',
#         'early_stopping_rounds': 200,
#         'random_state': CONFIG.SEED + trial.number,
#         'n_jobs': -1,
#         'verbosity': 0,
#         'enable_categorical': False,
#     }
    
#     # GPU configuration
#     if booster == 'gbtree':
#         params['tree_method'] = 'hist'
#         # if booster == 'gbtree':
#         params['device'] = 'cuda'

#     if booster == 'dart':
#         params['tree_method'] = 'hist'
#         params['device'] = 'cpu'
    

    
#     print(f"Params: LR={learning_rate:.5f}, n_est={n_estimators}, depth={max_depth}")
#     print(f"Reg: alpha={reg_alpha:.3e}, lambda={reg_lambda:.3e}, gamma={gamma:.3f}")
#     print(f"Sampling: subsample={subsample:.2f}, colsample={colsample_bytree:.2f}")
    
#     # ===== K-FOLD TRAINING =====
#     oof_preds = np.zeros(len(X))
#     test_preds = np.zeros(len(X_test))
#     fold_scores = []
#     fold_times = []
    
#     for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#         fold_start = time.time()
        
#         X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
#         y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
#         X_test_fold = X_test.copy()
        
#         # ===== TARGET ENCODING =====
#         print(f"  Fold {fold}: Encoding...", end=" ")
#         for c in CATS:
#             TE = TargetEncoder(cv=5, random_state=CONFIG.SEED + fold + trial.number, shuffle=True)
#             X_train_fold[c] = TE.fit_transform(pd.DataFrame(X_train_fold[c]), y_train_fold).flatten()
#             X_val_fold[c] = TE.transform(pd.DataFrame(X_val_fold[c])).flatten()
#             X_test_fold[c] = TE.transform(pd.DataFrame(X_test[c])).flatten()
        
#         # ===== MODEL TRAINING =====
#         print("Training...", end=" ")
#         model = xgb.XGBClassifier(**params)
        
#         model.fit(
#             X_train_fold, y_train_fold,
#             eval_set=[(X_val_fold, y_val_fold)],
#             verbose=False
#         )
        
#         # ===== PREDICTIONS =====
#         val_preds = model.predict_proba(X_val_fold)[:, 1]
#         oof_preds[val_idx] = val_preds
#         test_preds += model.predict_proba(X_test_fold)[:, 1] / CONFIG.N_FOLDS
        
#         # ===== SCORE CALCULATION =====
#         fold_score = roc_auc_score(y_val_fold, val_preds)
#         fold_scores.append(fold_score)
#         fold_time = time.time() - fold_start
#         fold_times.append(fold_time)
        
#         print(f"Score: {fold_score:.6f} (Time: {fold_time:.1f}s)")
    
#     # ===== FINAL SCORE =====
#     oof_score = roc_auc_score(y, oof_preds)
#     trial_time = time.time() - trial_start
#     avg_fold_time = np.mean(fold_times)
    
#     # ===== STORE RESULTS =====
#     trial_predictions[trial.number] = {
#         'test_preds': test_preds.copy(),
#         'oof_score': oof_score,
#         'fold_scores': fold_scores.copy(),
#         'fold_times': fold_times.copy(),
#         'params': params.copy(),
#         'booster': booster,
#         'trial_time': trial_time,
#         'avg_fold_time': avg_fold_time,
#         'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
#     }
#     trial_oof_preds[trial.number] = oof_preds.copy()
    
#     # Store in info list
#     all_models_info.append({
#         'trial_number': trial.number,
#         'booster': booster,
#         'oof_score': oof_score,
#         'learning_rate': learning_rate,
#         'n_estimators': n_estimators,
#         'max_depth': max_depth,
#         'reg_alpha': reg_alpha,
#         'reg_lambda': reg_lambda,
#         'subsample': subsample,
#         'colsample_bytree': colsample_bytree,
#         'trial_time': trial_time,
#         'avg_fold_time': avg_fold_time,
#         'timestamp': trial_predictions[trial.number]['timestamp']
#     })
    
#     completed_trials += 1
    
#     # ===== LOGGING =====
#     print(f"\n{'='*70}")
#     print(f"TRIAL {trial.number} COMPLETE")
#     print(f"{'='*70}")
#     print(f"OOF Score: {oof_score:.6f}")
#     print(f"Fold scores: {[f'{s:.6f}' for s in fold_scores]}")
#     print(f"Times - Trial: {trial_time:.1f}s, Avg fold: {avg_fold_time:.1f}s")
#     print(f"Booster: {booster}, LR: {learning_rate:.5f}, Depth: {max_depth}")
#     print(f"Stored predictions for ensemble building")
    
#     # Save artifacts
#     save_optuna_artifacts(force_save=False)
    
#     return oof_score

# # ===== OPTUNA STUDY SETUP =====
# print(f"\n{'='*70}")
# print("OPTUNA HYPERPARAMETER OPTIMIZATION")
# print(f"{'='*70}")
# print(f"Target: Maximize ROC AUC")
# print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
# print(f"CV: {CONFIG.N_FOLDS}-fold with multicat stratification")
# print(f"Artifacts will be saved to: {ARTIFACT_DIR}")
# print(f"{'='*70}")

# # Create study WITHOUT pruner
# study = optuna.create_study(
#     direction='maximize',
#     study_name=f'xgb_heart_disease_no_prune_{int(time.time())}',
#     sampler=optuna.samplers.TPESampler(
#         seed=CONFIG.SEED,
#         multivariate=True,
#         n_startup_trials=1  # Random search first 10 trials
#     ),
#     pruner=None  # NO PRUNING - we want all trials for ensemble
# )

# # ===== OPTIMIZATION EXECUTION =====
# total_start_time = time.time()
# n_target_trials = 60  # Target 100 trials for good ensemble
# timeout_seconds = 18000  # 24 hours timeout

# print(f"\nStarting optimization...")
# print(f"Target: {n_target_trials} trials")
# print(f"Timeout: {timeout_seconds/3600:.1f} hours")
# print(f"{'='*70}")

# try:
#     study.optimize(
#         objective,
#         n_trials=n_target_trials,
#         timeout=timeout_seconds,
#         show_progress_bar=True
#     )
# except KeyboardInterrupt:
#     print("\n⚠ Optimization interrupted by user")
# except Exception as e:
#     print(f"\n⚠ Optimization error: {e}")
# finally:
#     # Always save final artifacts
#     print(f"\n{'='*70}")
#     print("FINALIZING OPTIMIZATION")
#     print(f"{'='*70}")
    
#     # Force final save
#     save_optuna_artifacts(force_save=True)
    
#     # Save study
#     joblib.dump(study, ARTIFACT_DIR / 'study_final.pkl')
    
#     # Print summary
#     total_time = time.time() - total_start_time
#     print(f"\nOPTIMIZATION SUMMARY")
#     print(f"{'='*70}")
#     print(f"Total trials completed: {len(trial_predictions)}")
#     print(f"Total time: {total_time/3600:.2f} hours")
#     print(f"Average trial time: {np.mean([t['trial_time'] for t in trial_predictions.values()]):.1f}s")
    
#     if trial_predictions:
#         # Find best trial
#         best_trial_num = max(trial_predictions.items(), key=lambda x: x[1]['oof_score'])[0]
#         best_score = trial_predictions[best_trial_num]['oof_score']
#         best_booster = trial_predictions[best_trial_num]['booster']
        
#         print(f"\nBEST TRIAL: #{best_trial_num}")
#         print(f"   Score: {best_score:.6f}")
#         print(f"   Booster: {best_booster}")
#         print(f"   Parameters:")
#         for key, value in trial_predictions[best_trial_num]['params'].items():
#             if key in ['learning_rate', 'reg_alpha', 'reg_lambda', 'gamma']:
#                 print(f"     {key}: {value:.6f}")
#             elif key in ['max_depth', 'n_estimators']:
#                 print(f"     {key}: {value}")
        
#         # Booster distribution
#         from collections import Counter
#         boosters = [t['booster'] for t in trial_predictions.values()]
#         booster_counts = Counter(boosters)
#         print(f"\n📊 BOOSTER DISTRIBUTION:")
#         for booster_type in ['gbtree', 'dart', 'gblinear']:
#             count = booster_counts.get(booster_type, 0)
#             if count > 0:
#                 booster_scores = [t['oof_score'] for t in trial_predictions.values() if t['booster'] == booster_type]
#                 avg_score = np.mean(booster_scores)
#                 print(f"   {booster_type}: {count} models, avg score: {avg_score:.6f}")
    
#     print(f"\nAll artifacts saved to: {ARTIFACT_DIR}")
#     print(f"Files saved:")
#     print(f"   - trial_predictions.pkl (all trial data)")
#     print(f"   - trial_oof_preds.pkl (OOF predictions)")
#     print(f"   - all_test_preds.npy (test predictions)")
#     print(f"   - all_oof_preds.npy (OOF predictions matrix)")
#     print(f"   - all_models_info.csv (summary CSV)")
#     print(f"   - metadata.json (metadata)")
#     print(f"   - study_final.pkl (optuna study)")
#     print(f"{'='*70}")
#     print("Optimization complete! Ready for ensemble creation.")

In [10]:
# ARTIFACT_DIR_CB = Path('/kaggle/working/optuna_artifacts_catboost')
# ARTIFACT_DIR_CB.mkdir(exist_ok=True)

# # Global storage for CatBoost trials
# trial_predictions_cb = {}
# trial_oof_preds_cb = {}
# trial_start_times_cb = {}
# all_models_info_cb = []
# completed_trials_cb = 0

# # ===== SAVING FUNCTIONS (copy from XGBoost but use CB dir) =====
# def save_optuna_artifacts_cb(force_save=False):
#     global completed_trials_cb
#     if not force_save and completed_trials_cb % 5 != 0:
#         return
#     try:
#         with open(ARTIFACT_DIR_CB / 'trial_predictions.pkl', 'wb') as f:
#             pickle.dump(trial_predictions_cb, f)
#         with open(ARTIFACT_DIR_CB / 'trial_oof_preds.pkl', 'wb') as f:
#             pickle.dump(trial_oof_preds_cb, f)
#         if all_models_info_cb:
#             pd.DataFrame(all_models_info_cb).to_csv(ARTIFACT_DIR_CB / 'all_models_info.csv', index=False)
#         if trial_predictions_cb:
#             trial_numbers = sorted(trial_predictions_cb.keys())
#             test_preds_list = [trial_predictions_cb[t]['test_preds'] for t in trial_numbers]
#             oof_preds_list = [trial_oof_preds_cb[t] for t in trial_numbers]
#             if test_preds_list:
#                 np.save(ARTIFACT_DIR_CB / 'all_test_preds.npy', np.stack(test_preds_list))
#                 np.save(ARTIFACT_DIR_CB / 'all_oof_preds.npy', np.stack(oof_preds_list))
#         metadata = {
#             'n_trials': len(trial_predictions_cb),
#             'completed_trials': completed_trials_cb,
#             'save_time': time.strftime('%Y-%m-%d %H:%M:%S'),
#             'best_trial': max(trial_predictions_cb.items(), key=lambda x: x[1]['oof_score'])[0] if trial_predictions_cb else None,
#             'best_score': max([t['oof_score'] for t in trial_predictions_cb.values()]) if trial_predictions_cb else 0
#         }
#         with open(ARTIFACT_DIR_CB / 'metadata.json', 'w') as f:
#             json.dump(metadata, f, indent=2)
#         if force_save or completed_trials_cb % 10 == 0:
#             print(f"💾 CatBoost artifacts saved for {len(trial_predictions_cb)} trials")
#     except Exception as e:
#         print(f"⚠ Failed to save CatBoost artifacts: {e}")

# # ===== CATBOOST OBJECTIVE FUNCTION =====
# def objective_catboost(trial):
#     global completed_trials_cb
#     trial_start = time.time()
#     trial_start_times_cb[trial.number] = trial_start

#     print(f"\n{'='*70}")
#     print(f"CATBOOST TRIAL {trial.number} STARTING")
#     print(f"{'='*70}")

#     # ===== PARAMETER SAMPLING =====
#     params = {
#         'iterations': trial.suggest_int('iterations', 2000, 10000, step=500),
#         'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
#         'depth': trial.suggest_int('depth', 4, 10),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
#         'border_count': trial.suggest_int('border_count', 32, 255),
#         'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
#         'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
#         'od_type': 'Iter',
#         'od_wait': trial.suggest_int('od_wait', 50, 200),
#         'eval_metric': 'AUC',
#         'random_seed': CONFIG.SEED + trial.number,
#         'verbose': False,
#         'task_type': 'GPU',
#         # 'devices': '0' if DEVICE == 'cuda' else None,
#     }

#     print(f"Params: iterations={params['iterations']}, LR={params['learning_rate']:.5f}, depth={params['depth']}")

#     # ===== K-FOLD TRAINING =====
#     oof_preds = np.zeros(len(X))
#     test_preds = np.zeros(len(X_test))
#     fold_scores = []
#     fold_times = []

#     for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#         fold_start = time.time()
#         X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
#         y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
#         X_test_fold = X_test.copy()

#         # Target encoding (same as before)
#         print(f"  Fold {fold}: Encoding...", end=" ")
#         for c in CATS:
#             TE = TargetEncoder(cv=5, random_state=CONFIG.SEED + fold + trial.number, shuffle=True)
#             X_train_fold[c] = TE.fit_transform(pd.DataFrame(X_train_fold[c]), y_train_fold).flatten()
#             X_val_fold[c] = TE.transform(pd.DataFrame(X_val_fold[c])).flatten()
#             X_test_fold[c] = TE.transform(pd.DataFrame(X_test[c])).flatten()

#         # Create CatBoost Pools
#         train_pool = Pool(X_train_fold, y_train_fold, feature_names=list(X_train_fold.columns))
#         val_pool = Pool(X_val_fold, y_val_fold, feature_names=list(X_val_fold.columns))
#         test_pool = Pool(X_test_fold, feature_names=list(X_test_fold.columns))

#         # Train
#         print("Training...", end=" ")
#         model = cb.CatBoostClassifier(**params)
#         model.fit(train_pool, eval_set=val_pool, verbose=False, early_stopping_rounds=params['od_wait'])

#         # Predict
#         val_preds = model.predict_proba(val_pool)[:, 1]
#         oof_preds[val_idx] = val_preds
#         test_preds += model.predict_proba(test_pool)[:, 1] / CONFIG.N_FOLDS

#         fold_score = roc_auc_score(y_val_fold, val_preds)
#         fold_scores.append(fold_score)
#         fold_time = time.time() - fold_start
#         fold_times.append(fold_time)
#         print(f"Score: {fold_score:.6f} (Time: {fold_time:.1f}s)")

#     oof_score = roc_auc_score(y, oof_preds)
#     trial_time = time.time() - trial_start
#     avg_fold_time = np.mean(fold_times)

#     # Store results
#     trial_predictions_cb[trial.number] = {
#         'test_preds': test_preds.copy(),
#         'oof_score': oof_score,
#         'fold_scores': fold_scores,
#         'fold_times': fold_times,
#         'params': params,
#         'trial_time': trial_time,
#         'avg_fold_time': avg_fold_time,
#         'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
#     }
#     trial_oof_preds_cb[trial.number] = oof_preds.copy()

#     all_models_info_cb.append({
#         'trial_number': trial.number,
#         'oof_score': oof_score,
#         'iterations': params['iterations'],
#         'learning_rate': params['learning_rate'],
#         'depth': params['depth'],
#         'trial_time': trial_time,
#         'timestamp': trial_predictions_cb[trial.number]['timestamp']
#     })

#     completed_trials_cb += 1

#     print(f"\n{'='*70}")
#     print(f"CATBOOST TRIAL {trial.number} COMPLETE")
#     print(f"{'='*70}")
#     print(f"OOF Score: {oof_score:.6f}")
#     print(f"Times: {trial_time:.1f}s total, {avg_fold_time:.1f}s avg fold")

#     save_optuna_artifacts_cb(force_save=False)
#     return oof_score

# # ===== OPTUNA STUDY =====
# study_cb = optuna.create_study(
#     direction='maximize',
#     study_name='catboost_optuna',
#     sampler=optuna.samplers.TPESampler(seed=CONFIG.SEED, multivariate=True, n_startup_trials=5),
#     pruner=None
# )

# print("\n🚀 Starting CatBoost optimization...")
# study_cb.optimize(objective_catboost, n_trials=100, timeout=38000, show_progress_bar=True)

# # Final save
# save_optuna_artifacts_cb(force_save=True)
# joblib.dump(study_cb, ARTIFACT_DIR_CB / 'study_final.pkl')
# print(f"\n✅ CatBoost artifacts saved to {ARTIFACT_DIR_CB}")

In [11]:
ARTIFACT_DIR_LGB = Path('/kaggle/working/optuna_artifacts_lightgbm')
ARTIFACT_DIR_LGB.mkdir(exist_ok=True)

trial_predictions_lgb = {}
trial_oof_preds_lgb = {}
trial_start_times_lgb = {}
all_models_info_lgb = []
completed_trials_lgb = 0

def save_optuna_artifacts_lgb(force_save=False):
    global completed_trials_lgb
    if not force_save and completed_trials_lgb % 5 != 0:
        return
    try:
        with open(ARTIFACT_DIR_LGB / 'trial_predictions.pkl', 'wb') as f:
            pickle.dump(trial_predictions_lgb, f)
        with open(ARTIFACT_DIR_LGB / 'trial_oof_preds.pkl', 'wb') as f:
            pickle.dump(trial_oof_preds_lgb, f)
        if all_models_info_lgb:
            pd.DataFrame(all_models_info_lgb).to_csv(ARTIFACT_DIR_LGB / 'all_models_info.csv', index=False)
        if trial_predictions_lgb:
            trial_numbers = sorted(trial_predictions_lgb.keys())
            test_preds_list = [trial_predictions_lgb[t]['test_preds'] for t in trial_numbers]
            oof_preds_list = [trial_oof_preds_lgb[t] for t in trial_numbers]
            if test_preds_list:
                np.save(ARTIFACT_DIR_LGB / 'all_test_preds.npy', np.stack(test_preds_list))
                np.save(ARTIFACT_DIR_LGB / 'all_oof_preds.npy', np.stack(oof_preds_list))
        metadata = {
            'n_trials': len(trial_predictions_lgb),
            'completed_trials': completed_trials_lgb,
            'save_time': time.strftime('%Y-%m-%d %H:%M:%S'),
            'best_trial': max(trial_predictions_lgb.items(), key=lambda x: x[1]['oof_score'])[0] if trial_predictions_lgb else None,
            'best_score': max([t['oof_score'] for t in trial_predictions_lgb.values()]) if trial_predictions_lgb else 0
        }
        with open(ARTIFACT_DIR_LGB / 'metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        if force_save or completed_trials_lgb % 10 == 0:
            print(f"💾 LightGBM artifacts saved for {len(trial_predictions_lgb)} trials")
    except Exception as e:
        print(f"⚠ Failed to save LightGBM artifacts: {e}")

def objective_lightgbm(trial):
    global completed_trials_lgb
    trial_start = time.time()
    trial_start_times_lgb[trial.number] = trial_start

    print(f"\n{'='*70}")
    print(f"LIGHTGBM TRIAL {trial.number} STARTING")
    print(f"{'='*70}")

    # Base parameters
    boosting_type = trial.suggest_categorical('boosting_type', ['goss'])

    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': boosting_type,
        'num_leaves': trial.suggest_int('num_leaves', 31, 256), # Expanded for 600k rows
        'max_depth': trial.suggest_int('max_depth', 5, 15),     # Prevent "runaway" trees
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.9),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'verbose': -1,
        'n_jobs': -1,
        'random_state': CONFIG.SEED + trial.number,
        'device': 'cpu',
    }
    # params = {
    #     'objective': 'binary',
    #     'metric': 'auc',
    #     'boosting_type': boosting_type,
    #     'num_leaves': trial.suggest_int('num_leaves', 31, 256),
    #     'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
    #     'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
    #     'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
    #     'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
    #     'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    #     'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
    #     'verbose': -1,
    #     'n_jobs': -1,
    #     'random_state': CONFIG.SEED + trial.number,
    #     'device': 'gpu' ,
    # }

    # GOSS does not use bagging; use its own sampling parameters
    if boosting_type == 'goss':
            # GOSS optimization
            params['top_rate'] = trial.suggest_float('top_rate', 0.1, 0.4)
            params['other_rate'] = trial.suggest_float('other_rate', 0.05, 0.2)
    else:
            # Standard GBDT bagging
            params['bagging_fraction'] = trial.suggest_float('bagging_fraction', 0.5, 0.9)
            params['bagging_freq'] = trial.suggest_int('bagging_freq', 1, 7)
    
        # Use a fixed high n_estimators and rely on EARLY STOPPING
        # n_estimators = 10000

    # DART-specific parameters
    if boosting_type == 'dart':
        params['drop_rate'] = trial.suggest_float('drop_rate', 0.01, 0.5)
        params['max_drop'] = trial.suggest_int('max_drop', 10, 50)
        params['skip_drop'] = trial.suggest_float('skip_drop', 0.01, 0.5)

    # GPU setup
    if params['device'] == 'gpu':
        params['gpu_platform_id'] = 0
        params['gpu_device_id'] = 0

    n_estimators = trial.suggest_int('n_estimators', 1000, 7000, step=500)

    print(f"Params: boosting={boosting_type}, LR={params['learning_rate']:.5f}, num_leaves={params['num_leaves']}")

    # ===== K-FOLD TRAINING =====
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_scores = []
    fold_times = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        fold_start = time.time()
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        X_test_fold = X_test.copy()

        # Target encoding (same as before)
        print(f"  Fold {fold}: Encoding...", end=" ")
        for c in CATS:
            TE = TargetEncoder(cv=5, random_state=CONFIG.SEED + fold + trial.number, shuffle=True)
            X_train_fold[c] = TE.fit_transform(pd.DataFrame(X_train_fold[c]), y_train_fold).flatten()
            X_val_fold[c] = TE.transform(pd.DataFrame(X_val_fold[c])).flatten()
            X_test_fold[c] = TE.transform(pd.DataFrame(X_test[c])).flatten()

        # Create LightGBM Datasets
        train_data = lgb.Dataset(X_train_fold, label=y_train_fold)
        val_data = lgb.Dataset(X_val_fold, label=y_val_fold, reference=train_data)

        # Train
        print("Training...", end=" ")
        model = lgb.train(
            params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=n_estimators,
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )

        # Predict
        val_preds = model.predict(X_val_fold, num_iteration=model.best_iteration)
        oof_preds[val_idx] = val_preds
        test_preds += model.predict(X_test_fold, num_iteration=model.best_iteration) / CONFIG.N_FOLDS

        fold_score = roc_auc_score(y_val_fold, val_preds)
        fold_scores.append(fold_score)
        fold_time = time.time() - fold_start
        fold_times.append(fold_time)
        print(f"Score: {fold_score:.6f} (Time: {fold_time:.1f}s)")

    oof_score = roc_auc_score(y, oof_preds)
    trial_time = time.time() - trial_start
    avg_fold_time = np.mean(fold_times)

    # Store results
    trial_predictions_lgb[trial.number] = {
        'test_preds': test_preds.copy(),
        'oof_score': oof_score,
        'fold_scores': fold_scores,
        'fold_times': fold_times,
        'params': params,
        'n_estimators': n_estimators,
        'trial_time': trial_time,
        'avg_fold_time': avg_fold_time,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    trial_oof_preds_lgb[trial.number] = oof_preds.copy()

    all_models_info_lgb.append({
        'trial_number': trial.number,
        'oof_score': oof_score,
        'boosting_type': boosting_type,
        'learning_rate': params['learning_rate'],
        'num_leaves': params['num_leaves'],
        'n_estimators': n_estimators,
        'trial_time': trial_time,
        'timestamp': trial_predictions_lgb[trial.number]['timestamp']
    })

    completed_trials_lgb += 1

    print(f"\n{'='*70}")
    print(f"LIGHTGBM TRIAL {trial.number} COMPLETE")
    print(f"{'='*70}")
    print(f"OOF Score: {oof_score:.6f}")
    print(f"Times: {trial_time:.1f}s total, {avg_fold_time:.1f}s avg fold")

    save_optuna_artifacts_lgb(force_save=False)
    return oof_score

# ===== OPTUNA STUDY =====
study_lgb = optuna.create_study(
    direction='maximize',
    study_name='lightgbm_optuna',
    sampler=optuna.samplers.TPESampler(seed=CONFIG.SEED, multivariate=True, n_startup_trials=5),
    pruner=None
)

print("\n🚀 Starting LightGBM optimization...")
study_lgb.optimize(objective_lightgbm, n_trials=45, timeout=18000, show_progress_bar=True)

# Final save
save_optuna_artifacts_lgb(force_save=True)
joblib.dump(study_lgb, ARTIFACT_DIR_LGB / 'study_final.pkl')
print(f"\n✅ LightGBM artifacts saved to {ARTIFACT_DIR_LGB}")

[I 2026-02-24 08:26:29,303] A new study created in memory with name: lightgbm_optuna



🚀 Starting LightGBM optimization...


  0%|          | 0/45 [00:00<?, ?it/s]


LIGHTGBM TRIAL 0 STARTING
Params: boosting=goss, LR=0.04480, num_leaves=115
  Fold 1: Encoding... Training... Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[158]	valid_0's auc: 0.955728
Score: 0.955728 (Time: 26.9s)
  Fold 2: Encoding... Training... Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[151]	valid_0's auc: 0.95457
Score: 0.954570 (Time: 26.1s)
  Fold 3: Encoding... Training... Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[154]	valid_0's auc: 0.955472
Score: 0.955472 (Time: 26.3s)
  Fold 4: Encoding... Training... Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[239]	valid_0's auc: 0.954944
Score: 0.954944 (Time: 31.8s)
  Fold 5: Encoding... Training... Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[170]	valid_0's auc: 0.955872